# Dreamy Researcher — code cell example

This notebook is a small end-to-end test:

1. Python basics and persistent variables
2. Creating a pandas DataFrame
3. Summary statistics
4. A small scikit-learn model
5. Reusing variables from earlier cells
6. Catching an error

Run the cells top to bottom, or use **Run all**.


## 1. Python basics

The final expression in a code cell is displayed, similar to a notebook.


In [ ]:
# Basic Python — this cell defines variables used later
name = "researcher"
squares = [n ** 2 for n in range(1, 11)]

print(f"Hello, {name}!")
print("Squares:", squares)
print("Sum of squares:", sum(squares))

len(squares)

## 2. Create a small dataset

Everything below uses only NumPy and pandas.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 120

study_hours = rng.normal(5, 1.5, n).clip(0, 12)
sleep_hours = rng.normal(7, 1.0, n).clip(3, 10)
attendance = rng.integers(60, 101, n)

score = (
    6 * study_hours
    + 2 * sleep_hours
    + 0.25 * attendance
    + rng.normal(0, 8, n)
    - 5
).clip(0, 100)

df = pd.DataFrame({
    "study_hours": study_hours.round(2),
    "sleep_hours": sleep_hours.round(2),
    "attendance": attendance,
    "score": score.round(2),
    "passed": (score >= 60).astype(int),
})

print("Shape:", df.shape)
df.head()

## 3. Explore the data


In [ ]:
print(df.describe().round(2))
print("\nMissing values:")
print(df.isna().sum())
print("\nPass rate:", round(df["passed"].mean() * 100, 1), "%")

## 4. Train a quick model

This uses `passed` as the target and the three numeric features.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

features = ["study_hours", "sleep_hours", "attendance"]
X = df[features]
y = df["passed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print(classification_report(y_test, predictions, target_names=["fail", "pass"]))

pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

## 5. Check that Python state persists across cells

This cell uses `df`, `model`, and `features` from the previous cells.


In [ ]:
best_feature = features[model.feature_importances_.argmax()]

print("Rows in df:", len(df))
print("Best feature:", best_feature)
print("Best feature importance:", round(float(model.feature_importances_.max()), 3))

# Reuse a variable from the first cell
print("Hello again,", name)

## 6. Error handling test

The app should print a readable error and keep the rest of the notebook usable.


In [ ]:
try:
    1 / 0
except ZeroDivisionError as exc:
    print("Caught the expected error:", exc)

print("The notebook is still running.")

## Try the sidebar modules

A CSV version of this dataset (without the intermediate `score` column) is
available as `samples/study-scores.csv`.

1. Click **Load CSV…** and open `samples/study-scores.csv`.
2. Run **Preprocessing**.
3. Run **Visualization**.
4. Choose **SHAP** or **LIME** and click **Explain**.

You can also export the generated data from a code cell:

```python
df.to_csv("study-scores.csv", index=False)
print("Wrote study-scores.csv")
```
